# 10a. Proposed Model Ablation Study

This notebook investigates alternative input representations for the profile-conditioned semantic retrieval approach.

The original proposed model combines the user's historical preference profile with the synthetic query before encoding the resulting text using E5-base-v2. The ablation study examines whether retrieval performance is affected by the maximum input length and by the ordering of the profile and query.

Four configurations are evaluated:

1. Profile + Query, 256 tokens — original configuration.
2. Profile + Query, 512 tokens.
3. Query + Profile, 256 tokens.
4. Query + Profile, 512 tokens.

All configurations use the same E5-base-v2 encoder, the same product embeddings, the same FAISS index containing the complete Amazon Electronics product catalogue, and the same 5,559 evaluation examples and ground-truth products.

The purpose of this experiment is to determine whether alternative representations improve retrieval performance without changing the underlying semantic encoder or product retrieval infrastructure.

## 10a.1 Experimental Setup

The ablation study changes only the textual representation supplied to the semantic encoder. Product representations and the FAISS retrieval index remain fixed across all experiments.

The evaluation therefore isolates the effect of input length and the ordering of user preference information and current query intent.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss
from transformers import AutoTokenizer, AutoModel


# Project paths
PROJECT_ROOT = Path("/users/adhp511/query-conditioned-recommender")
GENERATED_DIR = PROJECT_ROOT / "data" / "generated"

EVAL_DATA_PATH = GENERATED_DIR / "proposed_model_data.parquet"
FAISS_INDEX_PATH = GENERATED_DIR / "e5_faiss_product_index.bin"
E5_MODEL_PATH = Path.home() / "models" / "e5-base-v2"


# Load evaluation data
eval_data = pd.read_parquet(EVAL_DATA_PATH)

print("Evaluation data shape:", eval_data.shape)


# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


# Load tokenizer and E5 model
tokenizer = AutoTokenizer.from_pretrained(
    E5_MODEL_PATH,
    local_files_only=True
)

e5_model = AutoModel.from_pretrained(
    E5_MODEL_PATH,
    local_files_only=True
).to(DEVICE)

e5_model.eval()


# Load the existing FAISS index
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

print("FAISS vectors:", faiss_index.ntotal)
print("FAISS dimension:", faiss_index.d)

/users/adhp511/.pyenv/versions/thesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Evaluation data shape: (5559, 9)
Device: cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 528.06it/s]


FAISS vectors: 1610012
FAISS dimension: 768


In [2]:
print("Tokenizer max length:", tokenizer.model_max_length)
print("Model max position embeddings:", e5_model.config.max_position_embeddings)

Tokenizer max length: 512
Model max position embeddings: 512


## 10a.1 Profile + Query with 512 Tokens

The original proposed model uses a maximum input length of 256 tokens when encoding the concatenated user preference profile and synthetic query.

This ablation increases the maximum input length to 512 tokens while keeping all other components unchanged. The same E5-base-v2 encoder, full Amazon Electronics product catalogue, FAISS index, evaluation examples and retrieval depth are used.

The purpose is to determine whether truncation at 256 tokens limits the effectiveness of the profile-conditioned representation.

In [4]:
# Reconstruct the profile-conditioned input

eval_data["profile_conditioned_query"] = (
    "User preference profile: "
    + eval_data["profile_text"].fillna("").astype(str)
    + "\n\nCurrent product search query: "
    + eval_data["synthetic_query"].fillna("").astype(str)
)

print("Profile-conditioned inputs:", len(eval_data))
print(
    "Missing/empty inputs:",
    eval_data["profile_conditioned_query"].isna().sum()
    + (eval_data["profile_conditioned_query"].str.strip() == "").sum()
)

print("\nExample:")
print(eval_data["profile_conditioned_query"].iloc[0])

Profile-conditioned inputs: 5559
Missing/empty inputs: 0

Example:
User preference profile: User Preference Profile

The user has primarily purchased products in these categories:

- Micro SD Cards
- Portable Bluetooth Speakers
- Internal Hard Drives

Products the user rated highly:

- Patriot LX Series 64GB High Speed Micro SDXC Class 10 UHS-I Transfer Speeds For Action Cameras, Phones, Tablets, and PCs
- The AquaAudio Cubo – Waterproof Bluetooth Wireless Speaker with Strong Suction Cup
- Western Digital 2TB WD Black Performance Internal Hard Drive HDD - 7200 RPM, SATA 6 Gb/s, 64 MB Cache, 3.5" - WD2003FZEX
- EBL Pack of 6 1000mAh NI-MH Home Phone Rechargeable Battery for HHRP105 HHR-P105 HHRP105A, HHR-P105A KX242
- CyberPower CP900AVR AVR UPS System, 900VA/560W, 10 Outlets, Mini-Tower

Most recent purchases:

- EBL Pack of 6 1000mAh NI-MH Home Phone Rechargeable Battery for HHRP105 HHR-P105 HHRP105A, HHR-P105A KX242
- CyberPower CP900AVR AVR UPS System, 900VA/560W, 10 Outlets, Mini-T

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss

from transformers import AutoTokenizer, AutoModel

# Project paths
PROJECT_ROOT = Path("/users/adhp511/query-conditioned-recommender")
DATA_DIR = PROJECT_ROOT / "data"

EVAL_PATH = DATA_DIR / "generated" / "proposed_model_data.parquet"
FAISS_PATH = DATA_DIR / "generated" / "e5_faiss_product_index.bin"
E5_PATH = Path.home() / "models" / "e5-base-v2"

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Load evaluation data
eval_data = pd.read_parquet(EVAL_PATH)

print("Evaluation data shape:", eval_data.shape)
print("Columns:", eval_data.columns.tolist())

# Load E5 tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(E5_PATH)
e5_model = AutoModel.from_pretrained(E5_PATH)
e5_model = e5_model.to(DEVICE)
e5_model.eval()

print("E5 model loaded.")

# Load existing FAISS index
faiss_index = faiss.read_index(str(FAISS_PATH))

print("FAISS vectors:", faiss_index.ntotal)
print("FAISS dimension:", faiss_index.d)

/users/adhp511/.pyenv/versions/thesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: Quadro RTX 8000
Evaluation data shape: (5559, 9)
Columns: ['sample_id', 'user_id', 'synthetic_query', 'profile_text', 'ground_truth_asin', 'product_type', 'intended_use', 'attributes_json', 'compatibility_json']


Loading weights: 100%|██████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 1105.28it/s]


E5 model loaded.
FAISS vectors: 1610012
FAISS dimension: 768


## 10a.2 Constructing Profile-Conditioned Inputs

The profile-conditioned representation combines the user's historical preference profile with the current synthetic shopping query.

The two components are concatenated into a single textual input before semantic encoding. This preserves the same representation strategy used in the original proposed model while allowing the maximum sequence length to be varied in the ablation experiments.

In [3]:
# Construct the profile-conditioned input

eval_data["profile_conditioned_query"] = (
    "User preference profile: "
    + eval_data["profile_text"].fillna("").astype(str)
    + "\n\nCurrent product search query: "
    + eval_data["synthetic_query"].fillna("").astype(str)
)

print("Total examples:", len(eval_data))
print(
    "Missing profiles:",
    eval_data["profile_text"].isna().sum()
)
print(
    "Missing synthetic queries:",
    eval_data["synthetic_query"].isna().sum()
)
print(
    "Missing/empty profile-conditioned inputs:",
    eval_data["profile_conditioned_query"].isna().sum()
    + (eval_data["profile_conditioned_query"].str.strip() == "").sum()
)

print("\nExample:")
print(eval_data["profile_conditioned_query"].iloc[0])

Total examples: 5559
Missing profiles: 0
Missing synthetic queries: 0
Missing/empty profile-conditioned inputs: 0

Example:
User preference profile: User Preference Profile

The user has primarily purchased products in these categories:

- Micro SD Cards
- Portable Bluetooth Speakers
- Internal Hard Drives

Products the user rated highly:

- Patriot LX Series 64GB High Speed Micro SDXC Class 10 UHS-I Transfer Speeds For Action Cameras, Phones, Tablets, and PCs
- The AquaAudio Cubo – Waterproof Bluetooth Wireless Speaker with Strong Suction Cup
- Western Digital 2TB WD Black Performance Internal Hard Drive HDD - 7200 RPM, SATA 6 Gb/s, 64 MB Cache, 3.5" - WD2003FZEX
- EBL Pack of 6 1000mAh NI-MH Home Phone Rechargeable Battery for HHRP105 HHR-P105 HHRP105A, HHR-P105A KX242
- CyberPower CP900AVR AVR UPS System, 900VA/560W, 10 Outlets, Mini-Tower

Most recent purchases:

- EBL Pack of 6 1000mAh NI-MH Home Phone Rechargeable Battery for HHRP105 HHR-P105 HHRP105A, HHR-P105A KX242
- CyberPowe

In [4]:
# Encode profile-conditioned inputs using a 512-token limit

MAX_LENGTH_ABLATION = 512
BATCH_SIZE = 32

profile_query_512_embeddings = []

for start in range(0, len(eval_data), BATCH_SIZE):
    batch = eval_data.iloc[start:start + BATCH_SIZE]

    texts = [
        f"query: {text}"
        for text in batch["profile_conditioned_query"]
    ]

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH_ABLATION,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = e5_model(**inputs)

    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    masked_embeddings = (
        outputs.last_hidden_state * attention_mask
    )

    sum_embeddings = masked_embeddings.sum(dim=1)
    sum_mask = attention_mask.sum(dim=1).clamp(min=1)

    embeddings = sum_embeddings / sum_mask

    embeddings = torch.nn.functional.normalize(
        embeddings,
        p=2,
        dim=1
    )

    profile_query_512_embeddings.append(
        embeddings.cpu().numpy().astype(np.float32)
    )

profile_query_512_embeddings = np.vstack(
    profile_query_512_embeddings
)

print("Embedding shape:", profile_query_512_embeddings.shape)

norms = np.linalg.norm(
    profile_query_512_embeddings,
    axis=1
)

print("Mean norm:", norms.mean())
print("Minimum norm:", norms.min())
print("Maximum norm:", norms.max())

Embedding shape: (5559, 768)
Mean norm: 1.0
Minimum norm: 0.9999999
Maximum norm: 1.0000001


In [5]:
# Retrieve Top-50 products from the shared FAISS index

TOP_K = 50

profile_query_512_scores, profile_query_512_indices = faiss_index.search(
    profile_query_512_embeddings,
    TOP_K
)

print("Scores shape:", profile_query_512_scores.shape)
print("Indices shape:", profile_query_512_indices.shape)

print("\nFirst example — Top-10 FAISS indices:")
print(profile_query_512_indices[0, :10])

print("\nFirst example — Top-10 similarity scores:")
print(profile_query_512_scores[0, :10])

Scores shape: (5559, 50)
Indices shape: (5559, 50)

First example — Top-10 FAISS indices:
[ 832718 1322993 1026746   16150   82864    2795  950472 1114953  153548
  763756]

First example — Top-10 similarity scores:
[0.834816   0.83332634 0.83269036 0.8321228  0.8319833  0.83181846
 0.83152646 0.83147514 0.83127904 0.83127534]


In [6]:
# Map FAISS indices back to Amazon product ASINs

CATALOGUE_PATH = DATA_DIR / "generated" / "full_product_catalogue.parquet"

catalogue_asins = pd.read_parquet(
    CATALOGUE_PATH,
    columns=["parent_asin"]
)["parent_asin"].to_numpy()

print("Catalogue ASIN count:", len(catalogue_asins))
print("FAISS index count:", faiss_index.ntotal)

# Convert FAISS indices to ASINs
profile_query_512_retrieved_asins = [
    catalogue_asins[row].tolist()
    for row in profile_query_512_indices
]

# Create retrieval dataframe
profile_query_512_retrievals = eval_data[
    [
        "sample_id",
        "user_id",
        "synthetic_query",
        "profile_text",
        "ground_truth_asin"
    ]
].copy()

profile_query_512_retrievals["retrieved_asins"] = (
    profile_query_512_retrieved_asins
)

profile_query_512_retrievals["retrieval_scores"] = [
    row.tolist()
    for row in profile_query_512_scores
]

print("Retrieval dataframe shape:", profile_query_512_retrievals.shape)

print("\nFirst example:")
print(
    profile_query_512_retrievals.iloc[0][
        [
            "sample_id",
            "synthetic_query",
            "ground_truth_asin",
            "retrieved_asins",
            "retrieval_scores"
        ]
    ]
)

Catalogue ASIN count: 1610012
FAISS index count: 1610012
Retrieval dataframe shape: (5559, 7)

First example:
sample_id                                                            0
synthetic_query      Looking for a high-speed micro sd card for my ...
ground_truth_asin                                           B007VKII6A
retrieved_asins      [B079NNV9TM, B07CZSDQ3G, B072Q8X5PX, B07CZRS6X...
retrieval_scores     [0.8348159790039062, 0.8333263397216797, 0.832...
Name: 0, dtype: object


In [7]:
# Save the 512-token profile-conditioned retrieval results

ABLATION_512_PATH = (
    DATA_DIR
    / "generated"
    / "proposed_ablation_profile_query_512.parquet"
)

profile_query_512_retrievals.to_parquet(
    ABLATION_512_PATH,
    index=False
)

print("Saved:", ABLATION_512_PATH)
print(
    "File size:",
    round(ABLATION_512_PATH.stat().st_size / (1024**2), 2),
    "MB"
)

# Basic validation
saved_512 = pd.read_parquet(ABLATION_512_PATH)

print("\nSaved shape:", saved_512.shape)
print("Unique sample IDs:", saved_512["sample_id"].nunique())
print(
    "All retrievals contain 50 ASINs:",
    saved_512["retrieved_asins"].apply(len).eq(50).all()
)
print(
    "All retrievals contain 50 scores:",
    saved_512["retrieval_scores"].apply(len).eq(50).all()
)

Saved: /users/adhp511/query-conditioned-recommender/data/generated/proposed_ablation_profile_query_512.parquet
File size: 4.74 MB

Saved shape: (5559, 7)
Unique sample IDs: 5559
All retrievals contain 50 ASINs: True
All retrievals contain 50 scores: True


## 10a.3 Evaluation of the 512-Token Ablation

The 512-token profile-conditioned retrieval results are evaluated against the same held-out ground-truth products used for the original proposed model.

The same Recall, Mean Reciprocal Rank (MRR), and Normalised Discounted Cumulative Gain (NDCG) metrics are used at the same retrieval cut-offs. This provides a controlled comparison between the original 256-token representation and the 512-token ablation.

In [8]:
# Metric functions

K_VALUES = [1, 5, 10, 20, 50]


def get_ground_truth_rank(retrieved_asins, ground_truth_asin):
    """
    Return the 1-based rank of the ground-truth product.
    Return None if it is not present in the retrieved list.
    """
    for rank, asin in enumerate(retrieved_asins, start=1):
        if asin == ground_truth_asin:
            return rank
    return None


def recall_at_k(retrieved_asins, ground_truth_asin, k):
    rank = get_ground_truth_rank(retrieved_asins, ground_truth_asin)
    return int(rank is not None and rank <= k)


def mrr_at_k(retrieved_asins, ground_truth_asin, k):
    rank = get_ground_truth_rank(retrieved_asins, ground_truth_asin)
    if rank is not None and rank <= k:
        return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved_asins, ground_truth_asin, k):
    rank = get_ground_truth_rank(retrieved_asins, ground_truth_asin)
    if rank is not None and rank <= k:
        return 1.0 / np.log2(rank + 1)
    return 0.0


# Calculate 512-token metrics
ablation_512_metrics = []

for k in K_VALUES:
    recall_values = []
    mrr_values = []
    ndcg_values = []

    for _, row in profile_query_512_retrievals.iterrows():
        retrieved = row["retrieved_asins"]
        ground_truth = row["ground_truth_asin"]

        recall_values.append(
            recall_at_k(retrieved, ground_truth, k)
        )

        mrr_values.append(
            mrr_at_k(retrieved, ground_truth, k)
        )

        ndcg_values.append(
            ndcg_at_k(retrieved, ground_truth, k)
        )

    ablation_512_metrics.append({
        "K": k,
        "Recall@K": np.mean(recall_values),
        "MRR@K": np.mean(mrr_values),
        "NDCG@K": np.mean(ndcg_values)
    })

ablation_512_metrics = pd.DataFrame(ablation_512_metrics)

print(ablation_512_metrics.to_string(index=False))

 K  Recall@K    MRR@K   NDCG@K
 1  0.000540 0.000540 0.000540
 5  0.002518 0.001280 0.001589
10  0.003958 0.001475 0.002057
20  0.005037 0.001547 0.002327
50  0.007735 0.001634 0.002865


In [10]:
# Load the original 256-token proposed-model retrievals

PROPOSED_256_PATH = (
    DATA_DIR
    / "generated"
    / "proposed_model_retrievals.parquet"
)

proposed_retrievals = pd.read_parquet(PROPOSED_256_PATH)

print("Original proposed model shape:", proposed_retrievals.shape)
print("Columns:", proposed_retrievals.columns.tolist())

Original proposed model shape: (5559, 7)
Columns: ['sample_id', 'user_id', 'synthetic_query', 'profile_text', 'ground_truth_asin', 'retrieved_asins', 'retrieval_scores']


In [11]:
# Compare Top-50 and Top-1 hit counts

def hit_count(retrievals, k):
    return sum(
        get_ground_truth_rank(
            row["retrieved_asins"],
            row["ground_truth_asin"]
        ) is not None
        and get_ground_truth_rank(
            row["retrieved_asins"],
            row["ground_truth_asin"]
        ) <= k
        for _, row in retrievals.iterrows()
    )


print("Profile + Query (256 tokens)")
print("Top-1 hits:", hit_count(proposed_retrievals, 1))
print("Top-50 hits:", hit_count(proposed_retrievals, 50))

print("\nProfile + Query (512 tokens)")
print("Top-1 hits:", hit_count(profile_query_512_retrievals, 1))
print("Top-50 hits:", hit_count(profile_query_512_retrievals, 50))

Profile + Query (256 tokens)
Top-1 hits: 5
Top-50 hits: 38

Profile + Query (512 tokens)
Top-1 hits: 3
Top-50 hits: 43


## 10a.4 Query + Profile with 512 Tokens

This ablation reverses the order of the two components in the profile-conditioned input.

The current synthetic query is placed before the historical user preference profile, while the maximum sequence length remains 512 tokens.

The purpose is to examine whether prioritising the current query during sequence truncation affects retrieval performance. All other components remain unchanged: E5-base-v2, the full Amazon Electronics catalogue, the shared FAISS index, Top-50 retrieval and the same 5,559 evaluation examples.

In [12]:
# Construct Query + Profile inputs

eval_data["query_profile_conditioned"] = (
    "Current product search query: "
    + eval_data["synthetic_query"].fillna("").astype(str)
    + "\n\nUser preference profile: "
    + eval_data["profile_text"].fillna("").astype(str)
)

print("Total examples:", len(eval_data))
print(
    "Missing/empty inputs:",
    eval_data["query_profile_conditioned"].isna().sum()
    + (eval_data["query_profile_conditioned"].str.strip() == "").sum()
)

print("\nExample:")
print(eval_data["query_profile_conditioned"].iloc[0])

Total examples: 5559
Missing/empty inputs: 0

Example:
Current product search query: Looking for a high-speed micro sd card for my action camera

User preference profile: User Preference Profile

The user has primarily purchased products in these categories:

- Micro SD Cards
- Portable Bluetooth Speakers
- Internal Hard Drives

Products the user rated highly:

- Patriot LX Series 64GB High Speed Micro SDXC Class 10 UHS-I Transfer Speeds For Action Cameras, Phones, Tablets, and PCs
- The AquaAudio Cubo – Waterproof Bluetooth Wireless Speaker with Strong Suction Cup
- Western Digital 2TB WD Black Performance Internal Hard Drive HDD - 7200 RPM, SATA 6 Gb/s, 64 MB Cache, 3.5" - WD2003FZEX
- EBL Pack of 6 1000mAh NI-MH Home Phone Rechargeable Battery for HHRP105 HHR-P105 HHRP105A, HHR-P105A KX242
- CyberPower CP900AVR AVR UPS System, 900VA/560W, 10 Outlets, Mini-Tower

Most recent purchases:

- EBL Pack of 6 1000mAh NI-MH Home Phone Rechargeable Battery for HHRP105 HHR-P105 HHRP105A, HHR-P

In [13]:
# Encode Query + Profile inputs using a 512-token limit

MAX_LENGTH_ABLATION = 512
BATCH_SIZE = 32

query_profile_512_embeddings = []

for start in range(0, len(eval_data), BATCH_SIZE):
    batch = eval_data.iloc[start:start + BATCH_SIZE]

    texts = [
        f"query: {text}"
        for text in batch["query_profile_conditioned"]
    ]

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH_ABLATION,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = e5_model(**inputs)

    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    masked_embeddings = (
        outputs.last_hidden_state * attention_mask
    )

    sum_embeddings = masked_embeddings.sum(dim=1)
    sum_mask = attention_mask.sum(dim=1).clamp(min=1)

    embeddings = sum_embeddings / sum_mask

    embeddings = torch.nn.functional.normalize(
        embeddings,
        p=2,
        dim=1
    )

    query_profile_512_embeddings.append(
        embeddings.cpu().numpy().astype(np.float32)
    )

query_profile_512_embeddings = np.vstack(
    query_profile_512_embeddings
)

print("Embedding shape:", query_profile_512_embeddings.shape)

norms = np.linalg.norm(
    query_profile_512_embeddings,
    axis=1
)

print("Mean norm:", norms.mean())
print("Minimum norm:", norms.min())
print("Maximum norm:", norms.max())

Embedding shape: (5559, 768)
Mean norm: 1.0
Minimum norm: 0.9999999
Maximum norm: 1.0000001


In [14]:
# Retrieve Top-50 products using Query + Profile (512 tokens)

TOP_K = 50

query_profile_512_scores, query_profile_512_indices = faiss_index.search(
    query_profile_512_embeddings,
    TOP_K
)

print("Scores shape:", query_profile_512_scores.shape)
print("Indices shape:", query_profile_512_indices.shape)

print("\nFirst example — Top-10 FAISS indices:")
print(query_profile_512_indices[0, :10])

print("\nFirst example — Top-10 similarity scores:")
print(query_profile_512_scores[0, :10])

Scores shape: (5559, 50)
Indices shape: (5559, 50)

First example — Top-10 FAISS indices:
[   2795  950472  185192   98174  548077  137260 1293980  818651  492507
   82864]

First example — Top-10 similarity scores:
[0.85969114 0.8574595  0.8513795  0.8510391  0.8509165  0.85030305
 0.8502959  0.84943306 0.8489635  0.8489437 ]


In [15]:
# Map FAISS indices back to Amazon product ASINs

catalogue_asins = pd.read_parquet(
    CATALOGUE_PATH,
    columns=["parent_asin"]
)["parent_asin"].to_numpy()

print("Catalogue ASIN count:", len(catalogue_asins))
print("FAISS index count:", faiss_index.ntotal)

# Convert FAISS indices to ASINs
query_profile_512_retrieved_asins = [
    catalogue_asins[row].tolist()
    for row in query_profile_512_indices
]

# Create retrieval dataframe
query_profile_512_retrievals = eval_data[
    [
        "sample_id",
        "user_id",
        "synthetic_query",
        "profile_text",
        "ground_truth_asin"
    ]
].copy()

query_profile_512_retrievals["retrieved_asins"] = (
    query_profile_512_retrieved_asins
)

query_profile_512_retrievals["retrieval_scores"] = [
    row.tolist()
    for row in query_profile_512_scores
]

print("Retrieval dataframe shape:", query_profile_512_retrievals.shape)

print("\nFirst example:")
print(
    query_profile_512_retrievals.iloc[0][
        [
            "sample_id",
            "synthetic_query",
            "ground_truth_asin",
            "retrieved_asins",
            "retrieval_scores"
        ]
    ]
)

Catalogue ASIN count: 1610012
FAISS index count: 1610012
Retrieval dataframe shape: (5559, 7)

First example:
sample_id                                                            0
synthetic_query      Looking for a high-speed micro sd card for my ...
ground_truth_asin                                           B007VKII6A
retrieved_asins      [B075B5T7L5, B01C9I5S1Q, B004HQD6HC, B0BPSZ7RW...
retrieval_scores     [0.8596911430358887, 0.8574594855308533, 0.851...
Name: 0, dtype: object


In [16]:
# Save the 512-token Query + Profile ablation

QUERY_PROFILE_512_PATH = (
    DATA_DIR
    / "generated"
    / "proposed_ablation_query_profile_512.parquet"
)

query_profile_512_retrievals.to_parquet(
    QUERY_PROFILE_512_PATH,
    index=False
)

print("Saved:", QUERY_PROFILE_512_PATH)
print(
    "File size:",
    round(QUERY_PROFILE_512_PATH.stat().st_size / (1024**2), 2),
    "MB"
)

# Validate saved file
saved_query_profile_512 = pd.read_parquet(
    QUERY_PROFILE_512_PATH
)

print("\nSaved shape:", saved_query_profile_512.shape)
print(
    "Unique sample IDs:",
    saved_query_profile_512["sample_id"].nunique()
)
print(
    "All retrievals contain 50 ASINs:",
    saved_query_profile_512["retrieved_asins"].apply(len).eq(50).all()
)
print(
    "All retrievals contain 50 scores:",
    saved_query_profile_512["retrieval_scores"].apply(len).eq(50).all()
)

Saved: /users/adhp511/query-conditioned-recommender/data/generated/proposed_ablation_query_profile_512.parquet
File size: 5.03 MB

Saved shape: (5559, 7)
Unique sample IDs: 5559
All retrievals contain 50 ASINs: True
All retrievals contain 50 scores: True


In [18]:
# Load the saved baseline and original proposed-model retrievals

BLAIR_PATH = DATA_DIR / "generated" / "blair_baseline_retrievals.parquet"
PROPOSED_PATH = DATA_DIR / "generated" / "proposed_model_retrievals.parquet"

query_only_retrievals = pd.read_parquet(BLAIR_PATH)
proposed_retrievals = pd.read_parquet(PROPOSED_PATH)

print("Query-only shape:", query_only_retrievals.shape)
print("Profile + Query (256) shape:", proposed_retrievals.shape)

print("\nQuery-only columns:")
print(query_only_retrievals.columns.tolist())

print("\nProfile + Query (256) columns:")
print(proposed_retrievals.columns.tolist())

Query-only shape: (5559, 6)
Profile + Query (256) shape: (5559, 7)

Query-only columns:
['sample_id', 'user_id', 'synthetic_query', 'ground_truth_asin', 'retrieved_asins', 'retrieval_scores']

Profile + Query (256) columns:
['sample_id', 'user_id', 'synthetic_query', 'profile_text', 'ground_truth_asin', 'retrieved_asins', 'retrieval_scores']


In [19]:
# Compare all four retrieval conditions

conditions = {
    "Query-only": query_only_retrievals,
    "Profile + Query (256)": proposed_retrievals,
    "Profile + Query (512)": profile_query_512_retrievals,
    "Query + Profile (512)": query_profile_512_retrievals,
}

comparison_results = []

for condition_name, retrievals in conditions.items():

    for k in K_VALUES:

        recall_values = []
        mrr_values = []
        ndcg_values = []

        for _, row in retrievals.iterrows():

            retrieved = row["retrieved_asins"]
            ground_truth = row["ground_truth_asin"]

            recall_values.append(
                recall_at_k(
                    retrieved,
                    ground_truth,
                    k
                )
            )

            mrr_values.append(
                mrr_at_k(
                    retrieved,
                    ground_truth,
                    k
                )
            )

            ndcg_values.append(
                ndcg_at_k(
                    retrieved,
                    ground_truth,
                    k
                )
            )

        comparison_results.append({
            "Condition": condition_name,
            "K": k,
            "Recall@K": np.mean(recall_values),
            "MRR@K": np.mean(mrr_values),
            "NDCG@K": np.mean(ndcg_values)
        })

comparison_results = pd.DataFrame(comparison_results)

print(comparison_results.to_string(index=False))

            Condition  K  Recall@K    MRR@K   NDCG@K
           Query-only  1  0.000000 0.000000 0.000000
           Query-only  5  0.000180 0.000045 0.000077
           Query-only 10  0.000540 0.000097 0.000198
           Query-only 20  0.001619 0.000176 0.000476
           Query-only 50  0.002698 0.000208 0.000684
Profile + Query (256)  1  0.000899 0.000899 0.000899
Profile + Query (256)  5  0.002518 0.001526 0.001774
Profile + Query (256) 10  0.003778 0.001695 0.002182
Profile + Query (256) 20  0.004497 0.001746 0.002365
Profile + Query (256) 50  0.006836 0.001827 0.002840
Profile + Query (512)  1  0.000540 0.000540 0.000540
Profile + Query (512)  5  0.002518 0.001280 0.001589
Profile + Query (512) 10  0.003958 0.001475 0.002057
Profile + Query (512) 20  0.005037 0.001547 0.002327
Profile + Query (512) 50  0.007735 0.001634 0.002865
Query + Profile (512)  1  0.000720 0.000720 0.000720
Query + Profile (512)  5  0.001799 0.001022 0.001209
Query + Profile (512) 10  0.002878 0.001164 0.

In [20]:
# Analyse tokenisation and potential truncation

texts = [
    f"query: {text}"
    for text in eval_data["profile_conditioned_query"]
]

token_lengths = []

for start in range(0, len(texts), 32):
    batch_texts = texts[start:start + 32]

    encoded = tokenizer(
        batch_texts,
        padding=False,
        truncation=False
    )

    token_lengths.extend(
        [len(ids) for ids in encoded["input_ids"]]
    )

token_lengths = np.array(token_lengths)

print("Number of examples:", len(token_lengths))
print("Mean token length:", token_lengths.mean())
print("Median token length:", np.median(token_lengths))
print("Minimum token length:", token_lengths.min())
print("Maximum token length:", token_lengths.max())

print("\nExamples exceeding 256 tokens:",
      np.sum(token_lengths > 256))

print("Percentage exceeding 256:",
      100 * np.mean(token_lengths > 256), "%")

print("\nExamples exceeding 512 tokens:",
      np.sum(token_lengths > 512))

print("Percentage exceeding 512:",
      100 * np.mean(token_lengths > 512), "%")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (547 > 512). Running this sequence through the model will result in indexing errors


Number of examples: 5559
Mean token length: 346.65569347004856
Median token length: 344.0
Minimum token length: 141
Maximum token length: 792

Examples exceeding 256 tokens: 4918
Percentage exceeding 256: 88.46914912754093 %

Examples exceeding 512 tokens: 70
Percentage exceeding 512: 1.2592192840438927 %


In [21]:
# Save token-length analysis for reporting

token_length_summary = pd.DataFrame({
    "Statistic": [
        "Number of examples",
        "Mean token length",
        "Median token length",
        "Minimum token length",
        "Maximum token length",
        "Examples exceeding 256 tokens",
        "Percentage exceeding 256 tokens",
        "Examples exceeding 512 tokens",
        "Percentage exceeding 512 tokens"
    ],
    "Value": [
        len(token_lengths),
        token_lengths.mean(),
        np.median(token_lengths),
        token_lengths.min(),
        token_lengths.max(),
        np.sum(token_lengths > 256),
        100 * np.mean(token_lengths > 256),
        np.sum(token_lengths > 512),
        100 * np.mean(token_lengths > 512)
    ]
})

print(token_length_summary.to_string(index=False))

                      Statistic       Value
             Number of examples 5559.000000
              Mean token length  346.655693
            Median token length  344.000000
           Minimum token length  141.000000
           Maximum token length  792.000000
  Examples exceeding 256 tokens 4918.000000
Percentage exceeding 256 tokens   88.469149
  Examples exceeding 512 tokens   70.000000
Percentage exceeding 512 tokens    1.259219
